# B3 multipassenger simulation

This notebook tests a shared travel graph, config-driven passenger batches, and route-level fleet sizing.


- Use one shared `TravelGraphManager` for all passengers.
- Export the graph through the HTML visualizer.
- Pull numeric values from `configs/travel_graph_config.yaml`.
- Sample passenger batch sizes from a mean and standard deviation.
- Keep fleet sizes per route flexible for later optimization work.


In [ ]:
from pathlib import Path
import sys
import importlib

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "configs" / "travel_graph_config.yaml").exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root (expected configs/travel_graph_config.yaml)."
    )

ROOT = find_repo_root(Path.cwd().resolve())

# Put repo root (and optional src/) on sys.path
for p in (ROOT, ROOT / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Single import surface: utils should export these in utils/__init__.py
utils = importlib.import_module("utils")

required = [
    "TravelGraphManager",
    "JeepneyRoute",
    "Passenger",
    "Simulation",
    "SimulationConfig",
]

missing = [name for name in required if not hasattr(utils, name)]
if missing:
    raise ImportError(
        "Missing exports in utils package: "
        f"{missing}. Add them to utils/__init__.py."
    )

TravelGraphManager = utils.TravelGraphManager
JeepneyRoute = utils.JeepneyRoute
Passenger = utils.Passenger
Simulation = utils.Simulation
SimulationConfig = utils.SimulationConfig

# Visualizer: allow one of the supported names, but fail loudly if none exists
for visualizer_name in (
    "TravelGraphVisualizer",
    "TravelGraphExplorer",
    "TravelGraphHtmlVisualizer",
):
    if hasattr(utils, visualizer_name):
        TravelGraphVisualizer = getattr(utils, visualizer_name)
        break
else:
    raise ImportError(
        "No visualizer export found in utils. "
        "Expected one of: TravelGraphVisualizer, "
        "TravelGraphExplorer, TravelGraphHtmlVisualizer."
    )

cfg = SimulationConfig.from_yaml(ROOT / "configs" / "travel_graph_config.yaml")

config_summary = {
    "walk_wt": cfg.walk_wt,
    "ride_wt": cfg.ride_wt,
    "wait_wt": cfg.wait_wt,
    "transfer_wt": cfg.transfer_wt,
    "v_jeep": cfg.v_jeep,
    "v_passenger": cfg.v_passenger,
    "headway_s": cfg.headway_s,
    "passenger_generation_interval_dt": cfg.passenger_generation_interval_dt,
    "passenger_generation_mean": cfg.passenger_generation_mean,
    "passenger_generation_std": cfg.passenger_generation_std,
}

ROOT, config_summary


{'walk_wt': 0.0142,
 'ride_wt': 0.0071,
 'wait_wt': 8.5,
 'transfer_wt': 14.2,
 'v_jeep': 10.0,
 'v_passenger': 1.4,
 'headway_s': 60.0,
 'passenger_generation_interval_dt': 2.0,
 'passenger_generation_mean': 120.0,
 'passenger_generation_std': 30.0}

In [ ]:
mgr = TravelGraphManager(
    edges_csv=ROOT / "data" / "iligan_travel_graph.csv",
    nodes_csv=ROOT / "data" / "travel_graph_nodes.csv",
    walk_wt=cfg.walk_wt,
    ride_wt=cfg.ride_wt,
    wait_wt=cfg.wait_wt,
    transfer_wt=cfg.transfer_wt,
)

visualizer = TravelGraphVisualizer(mgr, title="B3 Multipassenger Explorer")
html_path = visualizer.export(ROOT / "results" / "B3_multipassenger" / "interactive_travel_graph_explorer.html")
display(IFrame(str(html_path), width=1200, height=800))
html_path


In [ ]:
route_specs = [
    ("B3_R1", 2, 0),
    ("B3_R2", 3, 1),
    ("B3_R3", 4, 2),
    ("B3_R4", 5, 3),
]

routes = []
for route_id, min_len, seed in route_specs:
    loop = mgr.generate_random_ride_loop(min_len, min_len + 3, seed=seed)
    routes.append(JeepneyRoute(route_id, loop[:-1]))

sim = Simulation(
    travel_graph_mgr=mgr,
    passengers=[],
    routes=routes,
    config=cfg,
    fleet_sizes={"B3_R1": 2, "B3_R2": 3, "B3_R3": 4, "B3_R4": 5},
)

sim.summary()


In [ ]:
batch_sizes = [sim.sample_passenger_batch_size(seed=i) for i in range(4)]
batches = [sim.generate_passenger_batch(batch_size=size, seed=i) for i, size in enumerate(batch_sizes)]
all_passengers = [p for batch in batches for p in batch]
prepared = sim.prepare_passengers(all_passengers)

for step in range(6):
    sim.advance_fleets(sim_time=step * cfg.dt)

sample_report = {
    "batch_sizes": batch_sizes,
    "passengers_generated": len(all_passengers),
    "prepared_paths": sum(1 for item in prepared if item["found"]),
    "fleet_summary": sim.fleet_summary(),
    "total_jeeps": sim.total_jeeps,
    "cached_paths": len(sim._path_cache),
    "cached_nodes": len(sim._nearest_node_cache),
}
sample_report


## Notes

The shared graph and shared passenger map are the main scale controls.
Passenger batch size is sampled from config, so sensitivity runs can change the mean and standard deviation without touching the notebook logic.
